# FedFlow-LLM: Colab training run

Run cells top to bottom, one at a time. Never run a cell twice.

Use a GPU runtime (T4).

**Keep-alive (optional, not a cell):** press F12, open the Console tab on this Colab page, and paste this snippet so Colab doesn't disconnect while it's idle:

```javascript
function keepAlive() {
  // Click the "Connect"/status button every 60 s so Colab sees activity.
  const btn = document.querySelector("colab-connect-button");
  if (btn) btn.shadowRoot.querySelector("#connect").click();
}
setInterval(keepAlive, 60000);
```

In [ ]:
# Confirm a GPU is attached.
!nvidia-smi

In [ ]:
# Install dependencies (order matters, see README).
!pip install -q torch transformers peft accelerate bitsandbytes "flwr[simulation]"
!pip uninstall -y torch
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip uninstall -y torchvision torchaudio
!pip install -q transformers==4.46.3 peft==0.13.2

In [ ]:
# Clone the repo and move into training/, where every script runs.
!git clone https://github.com/shashank69420/fedflow-llm.git
%cd fedflow-llm/training

In [ ]:
# Generate the train/test data and check that no training window overlaps the test block.
!python generate_data.py && python check_split.py

SMOKE TEST: run the next cell, then STOP and report the output.

In [ ]:
# Smoke test: 1 training round, then inference on 5 held-out windows.
!FEDFLOW_ROUNDS=1 python server.py && python infer.py --eval --limit 5

In [ ]:
# Show the smoke-test output files and the metrics.
!ls -lh outputs
!cat outputs/metrics.json

DO NOT RUN BELOW until the smoke test has been checked.

In [ ]:
# Full run: 25 training rounds, then inference on all held-out windows.
!FEDFLOW_ROUNDS=25 python server.py && python infer.py --eval

In [ ]:
# Zip outputs/ and download it.
!zip -r outputs.zip outputs/
from google.colab import files
files.download("outputs.zip")